In [ ]:
!pip install git+https://github.com/salute-developers/GigaAM.git
!pip install datasets==3.6.0

# Augmentations

In [ ]:
import copy
import math
import os
import random
import subprocess

import datasets
import gigaam
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import sentencepiece
import soundfile
import torch
import torchaudio
from torch import nn

plt.style.use("dark_background")

In [ ]:
ds = datasets.load_dataset("fsicoli/common_voice_21_0", "ru", split="validation")

In [ ]:
N_SAMPLES = 100
SEED = 42
SAMPLE_RATE = 16_000

ds = ds.shuffle(SEED).select(range(N_SAMPLES))

In [ ]:
ds[0]

In [ ]:
ds = ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))

In [ ]:
gigaam_ctc_v2 = gigaam.load_model("v2_ctc", fp16_encoder=False, device="cpu")

In [ ]:
@torch.no_grad()
def get_gigaam_spectrogram(wav: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    wav = wav.astype(np.float32)
    device = next(gigaam_ctc_v2.encoder.parameters()).device

    wav_tensor = torch.frombuffer(wav, dtype=torch.float32).unsqueeze(0)
    wav_length = torch.full([1], wav_tensor.shape[-1])

    spectrogram, spec_len = gigaam_ctc_v2.preprocessor(
        wav_tensor.to(device),
        wav_length.to(device),
    )  # B x n_mels x T

    return spectrogram.cpu(), spec_len.cpu()


def plot_wav_with_features(
    wav: np.ndarray, spectrogram: np.ndarray, title: str | None = None
) -> None:

    plt.subplot(211)
    if title:
        plt.title(title)
    plt.plot(wav)
    plt.subplot(212)
    plt.imshow(spectrogram)
    plt.show()


for row in ds:
    print(row["sentence"])
    wav = row["audio"]["array"]
    spectrogram, spec_len = get_gigaam_spectrogram(wav)
    plot_wav_with_features(wav, spectrogram[0].numpy(), title="original")
    break

In [ ]:
class SpecAugment(torch.nn.Module):
    def __init__(
        self, time_mask_p=0.05, time_mask_param=30, freq_mask_param=15, num_masks=2
    ):
        super().__init__()
        self.time_mask_param = time_mask_param
        self.freq_mask_param = freq_mask_param
        self.num_masks = num_masks
        self.time_mask_p = time_mask_p

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        x = mel.clone()
        tm = torchaudio.transforms.TimeMasking(self.time_mask_param)
        fm = torchaudio.transforms.FrequencyMasking(self.freq_mask_param)
        for _ in range(self.num_masks):
            x = fm(x)
            if random.random() < self.time_mask_p:
                x = tm(x)
        return x


@torch.inference_mode()
def transcribe(spectrogram: torch.Tensor, spec_len: torch.Tensor) -> str:
    device = next(gigaam_ctc_v2.encoder.parameters()).device
    encoded, encoded_len = gigaam_ctc_v2.encoder(
        spectrogram.to(device), spec_len.to(device)
    )
    out = gigaam_ctc_v2.decoding.decode(gigaam_ctc_v2.head, encoded, encoded_len)
    return out[0]

In [ ]:
augmentor = SpecAugment(num_masks=5, time_mask_p=0.25, freq_mask_param=10)

for _ in range(3):
    augmented_spec = augmentor(spectrogram)
    hyp = transcribe(augmented_spec, spec_len)
    plot_wav_with_features(wav, augmented_spec[0].numpy(), title=f"SpecAugment: {hyp}")

In [ ]:
def opus_codec(wav: np.ndarray, sr: int, bitrate: str = "16k") -> np.ndarray:
    tmp_in = "_tmp_in.wav"
    tmp_out = "_tmp_out.wav"
    tmp_opus = "_tmp.opus"

    wav_int16 = (wav * 32767).astype(np.int16)

    soundfile.write(tmp_in, wav_int16, sr)

    try:
        subprocess.run(
            ["ffmpeg", "-i", tmp_in, "-c:a", "libopus", "-b:a", bitrate, tmp_opus],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        subprocess.run(
            ["ffmpeg", "-i", tmp_opus, "-ar", str(sr), tmp_out],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        y, _ = soundfile.read(tmp_out)
    finally:
        for p in [tmp_in, tmp_out, tmp_opus]:
            if os.path.exists(p):
                os.remove(p)
    return y

In [ ]:
for bitrate in ("128000", "8000", "500"):
    augmented_wav = opus_codec(wav, sr=SAMPLE_RATE, bitrate=bitrate)
    ipd.display(ipd.Audio(augmented_wav, rate=SAMPLE_RATE))
    augmented_spec, augmented_spec_len = get_gigaam_spectrogram(augmented_wav)
    hyp = transcribe(augmented_spec, augmented_spec_len)
    plot_wav_with_features(
        augmented_wav, augmented_spec[0].numpy(), title=f"Opus {bitrate}: {hyp}"
    )

In [ ]:
class NoiseAugment:
    def __init__(
        self,
        noise_ds_params: dict,
        target_snr_db: float = 10.0,
        sample_rate: int = 16_000,
        max_files: int = 100,
    ):
        super().__init__()
        self.target_snr_db = target_snr_db
        self.noises = [
            row["audio"]["array"]
            for row in (
                datasets.load_dataset(**noise_ds_params)
                .cast_column("audio", datasets.Audio(sampling_rate=sample_rate))
                .shuffle()
                .select(range(max_files))
            )
        ]

    def __call__(self, wav: np.ndarray) -> np.ndarray:
        noise = random.choice(self.noises)
        if len(noise) < len(wav):
            reps = math.ceil(len(wav) / len(noise))
            noise = np.tile(noise, reps)
        start = random.randint(0, len(noise) - len(wav))
        noise = noise[start : start + len(wav)]

        s_energy = np.mean(wav**2, dtype=np.float64) + 1e-12
        n_energy = np.mean(noise**2, dtype=np.float64) + 1e-12
        snr_linear = 10 ** (self.target_snr_db / 10)
        noise_scaled = noise * math.sqrt(s_energy / (snr_linear * n_energy))
        mixed = wav + noise_scaled

        return mixed

In [ ]:
# noise_ds_params = {
#     "path": "FluidInference/musan",
#     "split": "train",
# }


noise_ds_params = {
    "path": "fsicoli/common_voice_21_0",
    "name": "ky",
    "split": "other",
}

augmentor = NoiseAugment(noise_ds_params, target_snr_db=0)

for target_snr_db in (10, 5, 1):
    augmentor.target_snr_db = target_snr_db
    augmented_wav = augmentor(wav)
    ipd.display(ipd.Audio(augmented_wav, rate=SAMPLE_RATE))
    augmented_spec, augmented_spec_len = get_gigaam_spectrogram(augmented_wav)
    hyp = transcribe(augmented_spec, augmented_spec_len)
    plot_wav_with_features(
        augmented_wav,
        augmented_spec[0].numpy(),
        title=f"Noise SNR={target_snr_db}: {hyp}",
    )

# EncoderDecoder ASR Model

In [ ]:
VOCAB = 128
SPM_PREFIX = "spm_ky"
LANG = "ky"
DATASET = "fsicoli/common_voice_21_0"
SPLIT = "train+validation[:2%]"

if not os.path.exists(f"{SPM_PREFIX}.model"):

    ds = datasets.load_dataset(DATASET, LANG, split=SPLIT)

    ds_txt = ds.select_columns(["sentence"])
    with open("ky_corpus.txt", "w", encoding="utf-8") as f:
        for s in ds_txt["sentence"]:
            f.write(s.strip() + "\n")

    sentencepiece.SentencePieceTrainer.train(
        input="ky_corpus.txt",
        model_prefix=SPM_PREFIX,
        vocab_size=VOCAB,
        model_type="bpe",
        character_coverage=1.0,
        bos_id=1,
        eos_id=2,
        pad_id=3,
        unk_id=0,
    )

sp = sentencepiece.SentencePieceProcessor(model_file=f"{SPM_PREFIX}.model")
PAD, BOS, EOS = 3, 1, 2

ds = datasets.load_dataset(DATASET, LANG, split=SPLIT).cast_column(
    "audio", datasets.Audio(sampling_rate=SAMPLE_RATE)
)


def encode_text(s: str) -> torch.Tensor:
    ids = [BOS] + sp.encode(s or "") + [EOS]
    return torch.tensor(ids, dtype=torch.long)


def collate_fn(batch):
    features, features_length, targets, targets_length = [], [], [], []

    for ex in batch:
        wav = ex["audio"]["array"]
        spec, spec_len = get_gigaam_spectrogram(wav)
        text = encode_text(ex.get("sentence", ""))
        features.append(spec[0].T)
        features_length.append(spec_len)
        targets.append(text)
        targets_length.append(text.size(0))

    features_padded = torch.nn.utils.rnn.pad_sequence(
        features, batch_first=True
    ).permute(0, 2, 1)
    targets_padded = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True)

    return (
        features_padded,
        torch.Tensor(features_length).long(),
        targets_padded.long(),
        torch.Tensor(targets_length).long(),
    )


loader = torch.utils.data.DataLoader(
    ds, batch_size=8, shuffle=True, collate_fn=collate_fn
)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.pe = torch.nn.Parameter(pe, requires_grad=False)

    def forward(self, x):
        x = x + self.pe[: x.size(0), :]
        return self.dropout(x)


class DecoderLayerWithAttn(nn.Module):
    def __init__(self, d_model, n_heads, dim_ff, dropout=0.0, batch_first: bool = True):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=batch_first
        )
        self.cross_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=batch_first
        )
        self.lin1 = nn.Linear(d_model, dim_ff)
        self.lin2 = nn.Linear(dim_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.do1 = nn.Dropout(dropout)
        self.do2 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.act = nn.GELU()

    def forward(
        self,
        x,
        mem,
        tgt_mask,
        tgt_pad_mask,
        mem_pad_mask,
        need_weights=False,
        average_attn=True,
    ):
        # self-attn
        sa, _ = self.self_attn(
            x,
            x,
            x,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_pad_mask,
            need_weights=False,
        )
        x = self.norm1(x + self.do1(sa))
        # cross-attn
        ca, attn = self.cross_attn(
            x,
            mem,
            mem,
            key_padding_mask=mem_pad_mask,
            need_weights=need_weights,
            average_attn_weights=average_attn,
        )
        x = self.norm2(x + self.do2(ca))
        # FFN
        y = self.lin2(self.dropout(self.act(self.lin1(x))))
        x = self.norm3(x + y)
        return x, attn


class TransformerDecoder(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        embedding_dim: int,
        n_layers: int,
        tokenizer_path: str,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.tokenizer = sentencepiece.SentencePieceProcessor(model_file=tokenizer_path)

        self.embedding = nn.Embedding(self.tokenizer.vocab_size(), embedding_dim)
        self.pos_embedding = PositionalEncoding(embedding_dim, dropout=dropout)

        layer = DecoderLayerWithAttn(
            embedding_dim,
            n_heads,
            d_model,
            dropout,
            batch_first=True,
        )
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n_layers)])

        self.out = nn.Linear(embedding_dim, self.tokenizer.vocab_size())

    def forward(
        self,
        encoded: torch.Tensor,
        target: torch.Tensor,
        encoded_pad_mask: torch.Tensor | None = None,
        target_mask: torch.Tensor | None = None,
        target_pad_mask: torch.Tensor | None = None,
        return_attn: bool = False,
    ) -> tuple[torch.Tensor, list[torch.Tensor]]:
        emb = self.pos_embedding(self.embedding(target))
        out = emb
        attn_list = []
        for layer in self.layers:
            out, attn = layer(
                out,
                encoded,
                target_mask,
                target_pad_mask,
                encoded_pad_mask,
                need_weights=return_attn,
                average_attn=True,
            )
            if return_attn:
                attn_list.append(attn)
        out = self.out(out)
        return (out, attn_list)


class EncoderDecoder(nn.Module):
    def __init__(self, conf: dict):
        super().__init__()

        self.encoder = gigaam.load_model(
            conf["encoder"]["name"], fp16_encoder=False
        ).encoder
        self.decoder = TransformerDecoder(**conf["decoder"])
        self.projector = nn.Linear(
            conf["encoder"]["d_model"], self.decoder.embedding.embedding_dim
        )

        self.loss = nn.CrossEntropyLoss(reduction="none")

    @staticmethod
    def make_attention_mask(length: torch.Tensor) -> torch.Tensor:
        max_len = int(length.max().item())
        return torch.triu(
            torch.ones((max_len, max_len), device=length.device), diagonal=1
        ) * (-10000.0)

    @staticmethod
    def make_pad_mask(length: torch.Tensor) -> torch.Tensor:
        max_len = int(length.max().item())
        return torch.arange(0, max_len, device=length.device) < length.unsqueeze(1)

    def forward(
        self,
        features: torch.Tensor,
        features_len: torch.Tensor,
        targets: torch.Tensor,
        target_len: torch.Tensor,
        return_attn: bool = False,
    ) -> tuple[torch.Tensor, list[torch.Tensor]] | torch.Tensor:

        encoded, encoded_len = self.encoder(features, features_len)
        encoded = encoded.permute(0, 2, 1)
        encoded = self.projector(encoded)
        encoded_pad_mask = self.make_pad_mask(encoded_len)

        targets_outputs = targets[:, 1:]  # without bos
        targets_inputs = targets[:, :-1]  # without eos / last pad token
        target_len -= 1

        target_pad_mask = self.make_pad_mask(target_len)
        target_mask = self.make_attention_mask(target_len)

        logits, attn_list = self.decoder(
            encoded=encoded,
            encoded_pad_mask=~encoded_pad_mask,
            target=targets_inputs,
            target_mask=target_mask,
            target_pad_mask=~target_pad_mask,
            return_attn=return_attn,
        )

        loss = self.loss(logits.transpose(1, 2), targets_outputs)
        loss = (loss * target_pad_mask).sum() / target_pad_mask.sum()

        return (loss, attn_list) if return_attn else loss

In [ ]:
DEVICE = "cuda"

conf = {
    "encoder": {
        "name": "v2_ctc",
        "d_model": 768,
    },
    "decoder": {
        "d_model": 128,
        "n_heads": 4,
        "embedding_dim": 128,
        "n_layers": 2,
        "tokenizer_path": "./spm_ky.model",
    },
}

model = EncoderDecoder(conf).to(DEVICE)

dec_params = list(model.decoder.parameters()) + list(model.projector.parameters())

for p in model.encoder.parameters():
    p.requires_grad = False
model.encoder.eval()

opt = torch.optim.AdamW(dec_params, lr=1e-3)

N_EPOCHS = 3

model.train()
for _ in range(N_EPOCHS):
    for step, batch in enumerate(loader, 1):
        feats, flens, tgts, tlens = [x.to(DEVICE) for x in batch]
        loss = model(feats, flens, tgts, tlens)
        opt.zero_grad()
        loss.backward()
        opt.step()
        print(f"step {step}: loss {loss.item():.3f}")

        if step % 20 == 0:
            model.eval()
            with torch.no_grad():
                feats1, flens1, tgts1, tlens1 = (
                    feats[:1],
                    flens[:1],
                    tgts[:1],
                    tlens[:1],
                )

                feats_len_real = int(flens1.item())
                feats_in = feats1[:, :, :feats_len_real]
                tgt_len_real = int(tlens1.item())
                tgt_in = tgts1[:, :tgt_len_real]

                logits, attns = model.forward(
                    feats_in,
                    torch.tensor([feats_len_real], device=DEVICE),
                    tgt_in,
                    torch.tensor([tgt_len_real], device=DEVICE),
                    return_attn=True,
                )

                attn_last = attns[-1][0]
                plt.imshow(attn_last.cpu())
                plt.show()

            model.train()

In [ ]:
for p in model.encoder.parameters():
    p.requires_grad = True
model.train()


opt = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": 3e-4},
        {"params": dec_params, "lr": 1e-3},
    ],
    weight_decay=1e-2,
)

for step, batch in enumerate(loader, 1):
    feats, flens, tgts, tlens = [x.to(DEVICE) for x in batch]
    loss = model(feats, flens, tgts, tlens)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    print(f"step {step}: loss {loss.item():.3f}")
    if step % 20 == 0:
        model.eval()
        with torch.no_grad():
            feats1, flens1, tgts1, tlens1 = feats[:1], flens[:1], tgts[:1], tlens[:1]

            feats_len_real = int(flens1.item())
            feats_in = feats1[:, :, :feats_len_real]
            tgt_len_real = int(tlens1.item())
            tgt_in = tgts1[:, :tgt_len_real]

            logits, attns = model.forward(
                feats_in,
                torch.tensor([feats_len_real], device=DEVICE),
                tgt_in,
                torch.tensor([tgt_len_real], device=DEVICE),
                return_attn=True,
            )

            attn_last = attns[-1][0]
            plt.imshow(attn_last.cpu())
            plt.show()